# 📊 Day-4 · Notebook 05: RAG Evaluation with RAGAS Metrics & CI/CD Gates
**KIT Campus RAG Agent · Part 5: The RAG Triad & Automated CI/CD Testing**

In this notebook, we cover:
1. **The Lexical Overlap Failure**: Why BLEU and ROUGE fail on RAG evaluation.
2. **The RAG Triad**:
   - **Faithfulness**: Is the answer 100% grounded in retrieved context?
   - **Answer Relevancy**: Does the answer address the student's question?
   - **Context Precision**: Are the relevant chunks at the top of the search results?
   - **Context Recall**: Did the retrieval retrieve all needed facts?
3. **Automated CI/CD Quality Gates**: Enforcing test assertions in pipeline deployments (`assert faithfulness >= 0.85`).



In [ ]:
# 1. Why Word Overlap (BLEU/ROUGE) Fails on Semantic Paraphrasing
reference = "Students must submit the medical certificate within 3 calendar days."
candidate_exact = "Students must submit the medical certificate within 3 calendar days."
candidate_synonym = "A doctor certificate has to be turned in no later than 72 hours after returning."

def word_overlap(ref: str, cand: str) -> float:
    ref_tokens = set(ref.lower().replace(".", "").split())
    cand_tokens = set(cand.lower().replace(".", "").split())
    return len(ref_tokens.intersection(cand_tokens)) / len(ref_tokens)

print(f"Candidate Exact Overlap:   {word_overlap(reference, candidate_exact):.2f}")
print(f"Candidate Synonym Overlap: {word_overlap(reference, candidate_synonym):.2f}")
print("\nNOTICE: Synonym overlap is nearly 0.0 even though the meaning is 100% identical!\nThis is why modern RAG requires RAGAS LLM-as-a-Judge evaluation.")



In [ ]:
# 2. Automated RAGAS Evaluation Harness
class CampusRAGEvaluator:
    @staticmethod
    def evaluate(query: str, retrieved_context: str, generated_answer: str, ground_truth: str) -> Dict[str, float]:
        # Faithfulness check: check key factual claims
        faithfulness = 1.0 if ("65" in generated_answer and "₹500" in generated_answer) else 0.8
        # Answer Relevancy: check if the core question intent is addressed
        relevancy = 0.95 if ("condonation" in generated_answer.lower() or "attendance" in generated_answer.lower()) else 0.5
        # Precision & Recall
        precision = 1.0 if "medical condonation" in retrieved_context.lower() else 0.7
        recall = 1.0 if "₹500" in retrieved_context else 0.6

        return {
            "faithfulness": faithfulness,
            "answer_relevancy": relevancy,
            "context_precision": precision,
            "context_recall": recall
        }

eval_result = CampusRAGEvaluator.evaluate(
    query="What is the fee and timeline for medical condonation?",
    retrieved_context="Medical condonation (65%-74.9%) requires ₹500 fee per subject within 3 days.",
    generated_answer="Students must pay ₹500 per subject within 3 days of resuming classes [Doc 1].",
    ground_truth="Fee is ₹500 per subject, submit certificate within 3 days."
)

print("RAGAS Benchmark Evaluation Results:")
for metric, score in eval_result.items():
    status = "✓ PASS" if score >= 0.85 else "✗ FAIL"
    print(f"  - {metric.replace('_', ' ').title():<20}: {score:.2f} [{status}]")

